[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/09_sql-python.ipynb)

*Python for Data Science · Notebook 9 of 13*

# SQL from Python

Query a real database from Python: sqlite3 connections, safe parameters, joins, GROUP BY and HAVING, subqueries, window functions, dates, upserts, transactions, and moving data between SQL and pandas.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** The database is the warehouse and pandas is your desk: do the heavy sorting in the warehouse and carry only the small result to your desk.

**What interviewers really test here:** Joins and row multiplication, WHERE vs HAVING, window functions, parameterised queries against SQL injection, and when to aggregate in SQL vs pandas.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why analysts need SQL as well as pandas *(Basics)*
- [ ] 2. Parameters, and why f-strings are dangerous *(Basics)*
- [ ] 3. Joins and aggregation *(Core)*
- [ ] 4. Subqueries and window functions *(Core)*
- [ ] 5. Dates and pivots in SQL *(Core)*
- [ ] 6. SQL ↔ pandas, upserts, transactions and indexes *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Relational databases in one page

- A **table** has rows (records) and typed columns. A **primary key** identifies each row; a **foreign key** points at a row in another table (orders.customer\_id → customers.id).
- **Normalisation** stores each fact once (a customer's city lives in one place), and joins put the facts back together.
- **ACID**: transactions are Atomic (all or nothing), Consistent (constraints hold), Isolated (concurrent work doesn't interfere) and Durable (committed data survives crashes).
- **OLTP** databases (PostgreSQL, MySQL) handle many small reads and writes; **OLAP** warehouses (BigQuery, Snowflake, Redshift) scan huge tables for analytics.

### Join types

| Join | Keeps |
| --- | --- |
| INNER | only rows that match on both sides |
| LEFT | every left row; right columns NULL when unmatched |
| RIGHT / FULL OUTER | every right row / every row from both |
| CROSS | every combination (rows × rows) |
| Self join | a table joined to itself (employee → manager) |



If the join key isn't unique on both sides, rows multiply. Always check row counts before and after a join.

### NULL rules

- `NULL = NULL` is not true — use `IS NULL` / `IS NOT NULL`.
- `COUNT(*)` counts rows; `COUNT(col)` skips NULLs; `AVG`, `SUM` ignore NULLs.
- `COALESCE(col, 0)` replaces NULL with a default.
- `NOT IN (subquery)` returns nothing if the subquery contains a NULL; prefer `NOT EXISTS` or a LEFT JOIN.

### SQL vs pandas

| SQL | pandas |
| --- | --- |
| `WHERE a > 5` | `df[df.a > 5]` |
| `GROUP BY k` + `SUM(v)` | `df.groupby("k")["v"].sum()` |
| `HAVING` | filter after groupby |
| `JOIN` | `merge` |
| `ORDER BY … LIMIT n` | `nlargest` / `sort_values().head(n)` |
| `ROW_NUMBER() OVER (PARTITION BY k ORDER BY v)` | `groupby("k")["v"].rank(method="first")` |
| `SUM(v) OVER (PARTITION BY k ORDER BY t)` | `groupby("k")["v"].cumsum()` |

---
# Part 2: Lessons, step by step

## Lesson 1: Why analysts need SQL as well as pandas

*Level: Basics*

Company data lives in **databases**, not CSV files. pandas works on what fits in your laptop's memory; a database holds billions of rows and lets many people query them at once. The everyday workflow is: **use SQL to filter and aggregate inside the database, then pull the small result into pandas** for charts and modelling.



Python's built-in `sqlite3` module gives you a real SQL database in one file (or in memory) with nothing to install, so it's perfect for learning. The same SQL runs, with small dialect changes, on PostgreSQL, MySQL, BigQuery or Snowflake — you'd just connect with a different driver (`psycopg`, `sqlalchemy`).



```python
con = sqlite3.connect(":memory:")       # or a file: sqlite3.connect("shop.db")
con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", rows)
con.commit()
con.execute("SELECT COUNT(*) FROM orders").fetchone()   # (6,)
```



Want to practise pure SQL with automatic checking? The [SQL IDE](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/https://abpandey-dev.github.io/data-engineering-from-scratch-to-advanced/ide/sql.html) in the Data Engineering course has 43 checked questions.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
import sqlite3


def first_query():
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE orders (id INTEGER, city TEXT, amount REAL)")
    con.executemany("INSERT INTO orders VALUES (?, ?, ?)",
                    [(1, "Pune", 250.0), (2, "Delhi", 120.0), (3, "Pune", 90.0)])
    rows = con.execute("SELECT city, SUM(amount) FROM orders GROUP BY city ORDER BY city").fetchall()
    df = pd.read_sql_query("SELECT * FROM orders WHERE amount > ?", con, params=(100,))


first_query()

**✏️ Your turn:** you now have what you need for [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 2: Parameters, and why f-strings are dangerous

*Level: Basics*

Never build SQL by pasting user input into the text:



```python
name = "x' OR '1'='1"
con.execute(f"SELECT * FROM users WHERE name = '{name}'")      # returns EVERY user — SQL injection
con.execute("SELECT * FROM users WHERE name = ?", (name,))     # safe: the value is sent as data
```



With `?` placeholders the driver sends the query and the values separately, so quotes inside a value can never change the query. Parameters are also faster when the same query runs many times. (Table and column names can't be parameters — check them against an allow-list.)

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 3: Joins and aggregation

*Level: Core*

| Clause | Does | pandas equivalent |
| --- | --- | --- |
| `JOIN … ON` | rows that match in both tables | `merge(how="inner")` |
| `LEFT JOIN` | every left row; NULLs where nothing matches | `merge(how="left")` |
| `GROUP BY` | one row per group | `groupby` |
| `HAVING` | filter groups after aggregating | filter the grouped result |
| `CASE WHEN` | if / else inside a query | `np.select`, `pd.cut` |



The query runs in this order: **FROM/JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT**. That's why WHERE can't use a SUM (it doesn't exist yet) but HAVING can. To find "rows with no match", LEFT JOIN and keep rows where the right side `IS NULL`.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q10](#q10), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 4: Subqueries and window functions

*Level: Core*

A **subquery** is a query inside a query: `WHERE amount > (SELECT AVG(amount) FROM orders)`.



A **window function** computes across related rows but, unlike GROUP BY, keeps every row:



```python
ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC)   -- 1, 2, 3 within each customer
SUM(amount)  OVER (PARTITION BY customer_id ORDER BY day)           -- running total
LAG(amount)  OVER (ORDER BY day)                                   -- previous row's value
RANK() / DENSE_RANK()                                              -- ties share a rank
```



"Top N per group", "latest record per user", running totals and day-over-day change are all window-function questions — the most common SQL interview topic.

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q12](#q12), [Q13](#q13), [Q14](#q14), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 5: Dates and pivots in SQL

*Level: Core*

SQLite stores dates as ISO text (`'2026-02-14'`), which sorts correctly. `strftime('%Y-%m', day)` gives the month, `date(day, '+7 days')` shifts it, `julianday(b) - julianday(a)` gives days between. Other databases use `DATE_TRUNC('month', day)` or `EXTRACT`.



To turn months into columns (a pivot), use **conditional aggregation**:



```python
SELECT city,
       SUM(CASE WHEN strftime('%m', day) = '01' THEN amount ELSE 0 END) AS jan,
       SUM(CASE WHEN strftime('%m', day) = '02' THEN amount ELSE 0 END) AS feb
FROM orders JOIN customers ON … GROUP BY city
```

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 6: SQL ↔ pandas, upserts, transactions and indexes

*Level: Advanced*

```python
df = pd.read_sql_query("SELECT ...", con, params=(...))   # SQL → DataFrame
df.to_sql("sales", con, index=False, if_exists="append")    # DataFrame → table
```




- **Upsert**: insert, or update if the key already exists — `INSERT … ON CONFLICT(key) DO UPDATE SET …` (MERGE in other databases). It makes loads safe to re-run.
- **Transactions**: several statements that succeed or fail together. `with con:` commits at the end, or rolls back if an error is raised. Constraints like `CHECK (balance >= 0)` make the database refuse bad data.
- **Indexes**: a sorted lookup structure on a column, like the index at the back of a book. `WHERE customer_id = 1` becomes a fast search instead of reading every row. `EXPLAIN QUERY PLAN` shows which one happens. Indexes slow down writes a little and use space, so add them for columns you filter and join on.

**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q15](#q15), [Q16](#q16), [Q19](#q19), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [ ]:
import sqlite3


<a id="q1"></a>
### Q1. Load Rows and Count Them

🟢 Warm-up · *CREATE · executemany*

Load the two tables into SQLite and return `(number of customers, number of orders)`.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `(4, 6)`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi')], orders = []` → `(2, 0)`

In [ ]:
def counts(customers, orders):
    """
    Load the two tables into SQLite and return (number of customers, number of
    orders).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(counts, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), (4, 6)),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi')], []), (2, 0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Just count the Python lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build the database with `executemany` and ask it with `COUNT(*)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Just count the Python lists.

```python
def counts(customers, orders):
    return (len(customers), len(orders))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Build the database with `executemany` and ask it with `COUNT(*)`.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def counts(customers, orders):
    con = shop_db(customers, orders)
    c = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
    o = con.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    return (c, o)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Just count the Python lists. That costs O(1) time and O(1) space."

"I can do better: Build the database with `executemany` and ask it with `COUNT(*)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/load-rows-and-count-them)

<a id="q2"></a>
### Q2. Orders Above an Amount

🟢 Warm-up · *WHERE with a parameter*

Return `(id, amount)` for orders with amount ≥ `minimum`, highest first.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], minimum = 200` → `[(104, 400.0), (106, 310.0), (101, 250.0)]`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], minimum = 1000` → `[]`

In [ ]:
def big_orders(customers, orders, minimum):
    """
    Return (id, amount) for orders with amount ≥ minimum, highest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_orders, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 200), [(104, 400.0), (106, 310.0), (101, 250.0)]),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 1000), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Filter and sort the list in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`WHERE amount >= ?` with the value passed separately — never pasted into the SQL text.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Filter and sort the list in Python.

```python
def big_orders(customers, orders, minimum):
    rows = [(o[0], o[2]) for o in orders if o[2] >= minimum]
    return sorted(rows, key=lambda r: -r[1])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `WHERE amount >= ?` with the value passed separately — never pasted into the SQL text.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def big_orders(customers, orders, minimum):
    con = shop_db(customers, orders)
    sql = "SELECT id, amount FROM orders WHERE amount >= ? ORDER BY amount DESC"
    return con.execute(sql, (minimum,)).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Filter and sort the list in Python. That costs O(n log n) time and O(n) space."

"I can do better: `WHERE amount >= ?` with the value passed separately — never pasted into the SQL text. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/orders-above-an-amount)

<a id="q3"></a>
### Q3. Revenue per City

🟢 Warm-up · *JOIN + GROUP BY*

Return `(city, total amount)` for every city with orders, sorted by city.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `[('Delhi', 180.0), ('Pune', 1050.0)]`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05')]` → `[('Delhi', 120.0), ('Pune', 250.0)]`

In [ ]:
def city_revenue(customers, orders):
    """
    Return (city, total amount) for every city with orders, sorted by city.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_revenue, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), [('Delhi', 180.0), ('Pune', 1050.0)]),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05')]), [('Delhi', 120.0), ('Pune', 250.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Map customer → city in a dict, then add up amounts per city.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`JOIN` the tables and `GROUP BY city`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Map customer → city in a dict, then add up amounts per city.

```python
def city_revenue(customers, orders):
    city_of = {c[0]: c[2] for c in customers}
    totals = {}
    for o in orders:
        city = city_of[o[1]]
        totals[city] = totals.get(city, 0) + o[2]
    return sorted(totals.items())
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `JOIN` the tables and `GROUP BY city`.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def city_revenue(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT c.city, SUM(o.amount) FROM orders o
             JOIN customers c ON c.id = o.customer_id
             GROUP BY c.city ORDER BY c.city"""
    return con.execute(sql).fetchall()
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Map customer → city in a dict, then add up amounts per city. That costs O(n + m) time and O(n) space."

"I can do better: `JOIN` the tables and `GROUP BY city`. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/revenue-per-city)

<a id="q4"></a>
### Q4. Customers With No Orders

🟢 Warm-up · *LEFT JOIN … IS NULL*

Return the names of customers who never ordered, sorted.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `['John']`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = []` → `['Asha', 'John', 'Meera', 'Ravi']`

In [ ]:
def never_ordered(customers, orders):
    """
    Return the names of customers who never ordered, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(never_ordered, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), ['John']),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], []), ['Asha', 'John', 'Meera', 'Ravi']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A set of customer ids that ordered, then the rest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`LEFT JOIN` keeps every customer; missing orders show up as NULL.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A set of customer ids that ordered, then the rest.

```python
def never_ordered(customers, orders):
    buyers = {o[1] for o in orders}
    return sorted(c[1] for c in customers if c[0] not in buyers)
```

**Cost:** O(n + m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `LEFT JOIN` keeps every customer; missing orders show up as NULL.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def never_ordered(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT c.name FROM customers c LEFT JOIN orders o ON o.customer_id = c.id
             WHERE o.id IS NULL ORDER BY c.name"""
    return [r[0] for r in con.execute(sql)]
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A set of customer ids that ordered, then the rest. That costs O(n + m) time and O(m) space."

"I can do better: `LEFT JOIN` keeps every customer; missing orders show up as NULL. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/customers-with-no-orders)

<a id="q5"></a>
### Q5. Bucket Orders by Size

🟢 Warm-up · *CASE WHEN*

Label each order 'small' (&lt; 100), 'medium' (&lt; 300) or 'large', and return `(label, count)` sorted by label.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `[('large', 2), ('medium', 2), ('small', 2)]`

In [ ]:
def size_buckets(customers, orders):
    """
    Label each order 'small' (< 100), 'medium' (< 300) or 'large', and return
    (label, count) sorted by label.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(size_buckets, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), [('large', 2), ('medium', 2), ('small', 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain and a dict of counts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`CASE WHEN … END` inside the SELECT, then GROUP BY the label.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** An if-chain and a dict of counts.

```python
def size_buckets(customers, orders):
    counts = {}
    for o in orders:
        label = "small" if o[2] < 100 else "medium" if o[2] < 300 else "large"
        counts[label] = counts.get(label, 0) + 1
    return sorted(counts.items())
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `CASE WHEN … END` inside the SELECT, then GROUP BY the label.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def size_buckets(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT CASE WHEN amount < 100 THEN 'small' WHEN amount < 300 THEN 'medium' ELSE 'large' END AS label,
                    COUNT(*) FROM orders GROUP BY label ORDER BY label"""
    return con.execute(sql).fetchall()
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain and a dict of counts. That costs O(n) time and O(1) space."

"I can do better: `CASE WHEN … END` inside the SELECT, then GROUP BY the label. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/bucket-orders-by-size)

<a id="q6"></a>
### Q6. Monthly Revenue

🟢 Warm-up · *Dates with strftime*

Return `(month 'YYYY-MM', revenue, orders)` in month order.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `[('2026-01', 460.0, 3), ('2026-02', 770.0, 3)]`

In [ ]:
def monthly(customers, orders):
    """
    Return (month 'YYYY-MM', revenue, orders) in month order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(monthly, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), [('2026-01', 460.0, 3), ('2026-02', 770.0, 3)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Slice the first 7 characters of each date and add up.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`strftime('%Y-%m', day)` and GROUP BY it.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Slice the first 7 characters of each date and add up.

```python
def monthly(customers, orders):
    rev, cnt = {}, {}
    for o in orders:
        m = o[3][:7]
        rev[m] = rev.get(m, 0) + o[2]
        cnt[m] = cnt.get(m, 0) + 1
    return [(m, rev[m], cnt[m]) for m in sorted(rev)]
```

**Cost:** O(n) time, O(months) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `strftime('%Y-%m', day)` and GROUP BY it.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def monthly(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT strftime('%Y-%m', day) AS month, SUM(amount), COUNT(*) FROM orders
             GROUP BY month ORDER BY month"""
    return con.execute(sql).fetchall()
```

**Cost:** O(n) time, O(months) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Slice the first 7 characters of each date and add up. That costs O(n) time and O(months) space."

"I can do better: `strftime('%Y-%m', day)` and GROUP BY it. That's O(n) time and O(months) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/monthly-revenue)

<a id="q7"></a>
### Q7. Days With Several Customers

🟢 Warm-up · *COUNT(DISTINCT …)*

Return the days on which at least 2 different customers ordered.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `['2026-01-05']`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20'), (107, 4, 10.0, '2026-02-20')]` → `['2026-01-05', '2026-02-20']`

In [ ]:
def busy_days(customers, orders):
    """
    Return the days on which at least 2 different customers ordered.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(busy_days, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), ['2026-01-05']),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20'), (107, 4, 10.0, '2026-02-20')]), ['2026-01-05', '2026-02-20']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A set of customers per day.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`HAVING COUNT(DISTINCT customer_id) >= 2`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A set of customers per day.

```python
def busy_days(customers, orders):
    people = {}
    for o in orders:
        people.setdefault(o[3], set()).add(o[1])
    return sorted(d for d, s in people.items() if len(s) >= 2)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `HAVING COUNT(DISTINCT customer_id) >= 2`.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def busy_days(customers, orders):
    con = shop_db(customers, orders)
    sql = "SELECT day FROM orders GROUP BY day HAVING COUNT(DISTINCT customer_id) >= 2 ORDER BY day"
    return [r[0] for r in con.execute(sql)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A set of customers per day. That costs O(n) time and O(n) space."

"I can do better: `HAVING COUNT(DISTINCT customer_id) >= 2`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/days-with-several-customers)

<a id="q8"></a>
### Q8. Query Straight into a DataFrame

🟢 Warm-up · *pd.read\_sql\_query*

Return a DataFrame with columns `name` and `orders` (order count, including 0), sorted by name.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `{'name': ['Asha', 'John', 'Meera', 'Ravi'], 'orders': [3, 0, 1, 2]}`

In [ ]:
def orders_frame(customers, orders):
    """
    Return a DataFrame with columns name and orders (order count, including 0),
    sorted by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(orders_frame, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), {'name': ['Asha', 'John', 'Meera', 'Ravi'], 'orders': [3, 0, 1, 2]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in Python and build the DataFrame yourself.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.read_sql_query` runs the SQL and returns a DataFrame with the column names from the query.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count in Python and build the DataFrame yourself.

```python
def orders_frame(customers, orders):
    count = {c[0]: 0 for c in customers}
    for o in orders:
        count[o[1]] += 1
    rows = sorted((c[1], count[c[0]]) for c in customers)
    return pd.DataFrame(rows, columns=["name", "orders"])
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.read_sql_query` runs the SQL and returns a DataFrame with the column names from the query.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def orders_frame(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT c.name, COUNT(o.id) AS orders FROM customers c
             LEFT JOIN orders o ON o.customer_id = c.id GROUP BY c.id ORDER BY c.name"""
    return pd.read_sql_query(sql, con)
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in Python and build the DataFrame yourself. That costs O(n + m) time and O(n) space."

"I can do better: `pd.read_sql_query` runs the SQL and returns a DataFrame with the column names from the query. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/query-straight-into-a-dataframe)

<a id="q9"></a>
### Q9. Safe Lookup by Name

🟡 Interview standard · *SQL injection*

Return the ids of customers whose name equals `name` exactly. The test includes an attack string, which must match nobody.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], name = 'Ravi'` → `[2]`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], name = "x' OR '1'='1"` → `[]`

In [ ]:
def ids_named(customers, orders, name):
    """
    Return the ids of customers whose name equals name exactly. The test
    includes an attack string, which must match nobody.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ids_named, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 'Ravi'), [2]),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], "x' OR '1'='1"), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare names in Python — no SQL, so nothing to inject.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `?` placeholder: the driver sends the value as data, so quotes inside it can't change the query.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Compare names in Python — no SQL, so nothing to inject.

```python
def ids_named(customers, orders, name):
    return [c[0] for c in customers if c[1] == name]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A `?` placeholder: the driver sends the value as data, so quotes inside it can't change the query.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def ids_named(customers, orders, name):
    con = shop_db(customers, orders)
    rows = con.execute("SELECT id FROM customers WHERE name = ? ORDER BY id", (name,)).fetchall()
    return [r[0] for r in rows]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare names in Python — no SQL, so nothing to inject. That costs O(n) time and O(n) space."

"I can do better: A `?` placeholder: the driver sends the value as data, so quotes inside it can't change the query. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/safe-lookup-by-name)

<a id="q10"></a>
### Q10. Big Spenders

🟡 Interview standard · *HAVING*

Return `(name, total, number of orders)` for customers whose total spend is more than `limit`, biggest total first.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], limit = 300` → `[('Asha', 650.0, 3), ('Meera', 400.0, 1)]`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], limit = 600` → `[('Asha', 650.0, 3)]`

In [ ]:
def big_spenders(customers, orders, limit):
    """
    Return (name, total, number of orders) for customers whose total spend is
    more than limit, biggest total first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_spenders, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 300), [('Asha', 650.0, 3), ('Meera', 400.0, 1)]),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 600), [('Asha', 650.0, 3)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Total and count per customer in dicts, then filter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`GROUP BY` then `HAVING SUM(amount) > ?` — WHERE can't see sums, HAVING can.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Total and count per customer in dicts, then filter.

```python
def big_spenders(customers, orders, limit):
    name = {c[0]: c[1] for c in customers}
    total, count = {}, {}
    for o in orders:
        total[o[1]] = total.get(o[1], 0) + o[2]
        count[o[1]] = count.get(o[1], 0) + 1
    rows = [(name[k], total[k], count[k]) for k in total if total[k] > limit]
    return sorted(rows, key=lambda r: -r[1])
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `GROUP BY` then `HAVING SUM(amount) > ?` — WHERE can't see sums, HAVING can.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def big_spenders(customers, orders, limit):
    con = shop_db(customers, orders)
    sql = """SELECT c.name, SUM(o.amount) AS total, COUNT(*) FROM orders o
             JOIN customers c ON c.id = o.customer_id
             GROUP BY c.id HAVING SUM(o.amount) > ? ORDER BY total DESC"""
    return con.execute(sql, (limit,)).fetchall()
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Total and count per customer in dicts, then filter. That costs O(n + m) time and O(n) space."

"I can do better: `GROUP BY` then `HAVING SUM(amount) > ?` — WHERE can't see sums, HAVING can. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/big-spenders)

<a id="q11"></a>
### Q11. Orders Above Average

🟡 Interview standard · *Subquery*

Return the ids of orders whose amount is above the average order amount.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `[101, 104, 106]`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20')]` → `[101]`

In [ ]:
def above_average(customers, orders):
    """
    Return the ids of orders whose amount is above the average order amount.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(above_average, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), [101, 104, 106]),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20')]), [101]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two passes: compute the mean, then filter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A scalar subquery: `WHERE amount > (SELECT AVG(amount) FROM orders)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Two passes: compute the mean, then filter.

```python
def above_average(customers, orders):
    avg = sum(o[2] for o in orders) / len(orders)
    return [o[0] for o in orders if o[2] > avg]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A scalar subquery: `WHERE amount > (SELECT AVG(amount) FROM orders)`.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def above_average(customers, orders):
    con = shop_db(customers, orders)
    sql = "SELECT id FROM orders WHERE amount > (SELECT AVG(amount) FROM orders) ORDER BY id"
    return [r[0] for r in con.execute(sql)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two passes: compute the mean, then filter. That costs O(n) time and O(n) space."

"I can do better: A scalar subquery: `WHERE amount > (SELECT AVG(amount) FROM orders)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/orders-above-average)

<a id="q12"></a>
### Q12. Biggest Order per Customer

🟡 Interview standard · *ROW\_NUMBER() OVER (PARTITION BY …)*

Return `(customer_id, order id, amount)` of each customer's largest order, sorted by customer.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `[(1, 106, 310.0), (2, 102, 120.0), (3, 104, 400.0)]`

In [ ]:
def top_order(customers, orders):
    """
    Return (customer_id, order id, amount) of each customer's largest order,
    sorted by customer.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_order, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), [(1, 106, 310.0), (2, 102, 120.0), (3, 104, 400.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep the best order per customer in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Number the rows inside each customer by amount, keep row 1.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep the best order per customer in a dict.

```python
def top_order(customers, orders):
    best = {}
    for o in orders:
        if o[1] not in best or o[2] > best[o[1]][2]:
            best[o[1]] = (o[1], o[0], o[2])
    return [best[k] for k in sorted(best)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Number the rows inside each customer by amount, keep row 1.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def top_order(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT customer_id, id, amount FROM (
               SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rn FROM orders)
             WHERE rn = 1 ORDER BY customer_id"""
    return con.execute(sql).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep the best order per customer in a dict. That costs O(n) time and O(n) space."

"I can do better: Number the rows inside each customer by amount, keep row 1. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/biggest-order-per-customer)

<a id="q13"></a>
### Q13. Running Total per Customer

🟡 Interview standard · *SUM() OVER (ORDER BY …)*

For customer `cid`, return `(day, amount, running total)` in date order.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], cid = 1` → `[('2026-01-05', 250.0, 250.0), ('2026-01-20', 90.0, 340.0), ('2026-02-20', 310.0, 650.0)]`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], cid = 2` → `[('2026-01-05', 120.0, 120.0), ('2026-02-14', 60.0, 180.0)]`

In [ ]:
def running_total(customers, orders, cid):
    """
    For customer cid, return (day, amount, running total) in date order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(running_total, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 1), [('2026-01-05', 250.0, 250.0), ('2026-01-20', 90.0, 340.0), ('2026-02-20', 310.0, 650.0)]),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')], 2), [('2026-01-05', 120.0, 120.0), ('2026-02-14', 60.0, 180.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort that customer's orders and keep a running sum.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A window `SUM(amount) OVER (ORDER BY day)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Sort that customer's orders and keep a running sum.

```python
def running_total(customers, orders, cid):
    mine = sorted((o for o in orders if o[1] == cid), key=lambda o: o[3])
    out, total = [], 0
    for o in mine:
        total += o[2]
        out.append((o[3], o[2], total))
    return out
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A window `SUM(amount) OVER (ORDER BY day)`.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def running_total(customers, orders, cid):
    con = shop_db(customers, orders)
    sql = """SELECT day, amount, SUM(amount) OVER (ORDER BY day) FROM orders
             WHERE customer_id = ? ORDER BY day"""
    return con.execute(sql, (cid,)).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort that customer's orders and keep a running sum. That costs O(n log n) time and O(n) space."

"I can do better: A window `SUM(amount) OVER (ORDER BY day)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/running-total-per-customer)

<a id="q14"></a>
### Q14. Second-Highest Order Amount

🟡 Interview standard · *DISTINCT · LIMIT/OFFSET*

Return the second-highest distinct order amount, or None if there isn't one.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `310.0`
- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05')]` → `None`

In [ ]:
def second_highest(customers, orders):
    """
    Return the second-highest distinct order amount, or None if there isn't one.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(second_highest, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), 310.0),
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05')]), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A set of amounts, sorted descending.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SELECT DISTINCT … ORDER BY … LIMIT 1 OFFSET 1`, wrapped so an empty result becomes NULL.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A set of amounts, sorted descending.

```python
def second_highest(customers, orders):
    amounts = sorted({o[2] for o in orders}, reverse=True)
    return amounts[1] if len(amounts) > 1 else None
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `SELECT DISTINCT … ORDER BY … LIMIT 1 OFFSET 1`, wrapped so an empty result becomes NULL.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def second_highest(customers, orders):
    con = shop_db(customers, orders)
    sql = "SELECT (SELECT DISTINCT amount FROM orders ORDER BY amount DESC LIMIT 1 OFFSET 1)"
    return con.execute(sql).fetchone()[0]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A set of amounts, sorted descending. That costs O(n log n) time and O(n) space."

"I can do better: `SELECT DISTINCT … ORDER BY … LIMIT 1 OFFSET 1`, wrapped so an empty result becomes NULL. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/second-highest-order-amount)

<a id="q15"></a>
### Q15. From a DataFrame to a Table and Back

🟡 Interview standard · *df.to\_sql*

`data` holds columns product, qty and price. Make it a DataFrame, write it to a table, then return `(product, revenue)` rows from SQL, highest revenue first.

**Examples**

- `data = {'product': ['pen', 'book', 'bag'], 'qty': [10, 2, 1], 'price': [5.0, 120.0, 300.0]}` → `[('bag', 300.0), ('book', 240.0), ('pen', 50.0)]`

In [ ]:
def revenue_via_sql(data):
    """
    data holds columns product, qty and price. Make it a DataFrame, write it to
    a table, then return (product, revenue) rows from SQL, highest revenue
    first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(revenue_via_sql, [
    (({'product': ['pen', 'book', 'bag'], 'qty': [10, 2, 1], 'price': [5.0, 120.0, 300.0]},), [('bag', 300.0), ('book', 240.0), ('pen', 50.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute revenue in plain Python and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.to_sql` creates the table; SQL does the maths.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Compute revenue in plain Python and sort.

```python
def revenue_via_sql(data):
    rows = [(p, q * pr) for p, q, pr in zip(data["product"], data["qty"], data["price"])]
    return sorted(rows, key=lambda r: -r[1])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.to_sql` creates the table; SQL does the maths.

```python
def revenue_via_sql(data):
    df = pd.DataFrame(data)
    con = sqlite3.connect(":memory:")
    df.to_sql("sales", con, index=False)
    sql = "SELECT product, qty * price AS revenue FROM sales ORDER BY revenue DESC"
    return con.execute(sql).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute revenue in plain Python and sort. That costs O(n log n) time and O(n) space."

"I can do better: `df.to_sql` creates the table; SQL does the maths. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/from-a-dataframe-to-a-table-and-back)

<a id="q16"></a>
### Q16. Upsert Stock Levels

🟡 Interview standard · *INSERT … ON CONFLICT*

Start from `stock` (sku, qty). Apply `updates`: add the qty to an existing sku, or insert a new sku. Return all rows sorted by sku.

**Examples**

- `stock = [('A', 5), ('B', 2)], updates = [('B', 3), ('C', 7), ('A', -1)]` → `[('A', 4), ('B', 5), ('C', 7)]`

In [ ]:
def upsert(stock, updates):
    """
    Start from stock (sku, qty). Apply updates: add the qty to an existing sku,
    or insert a new sku. Return all rows sorted by sku.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(upsert, [
    (([('A', 5), ('B', 2)], [('B', 3), ('C', 7), ('A', -1)]), [('A', 4), ('B', 5), ('C', 7)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict: `get(sku, 0) + qty`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ON CONFLICT(sku) DO UPDATE SET qty = qty + excluded.qty`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A dict: `get(sku, 0) + qty`.

```python
def upsert(stock, updates):
    levels = dict(stock)
    for sku, qty in updates:
        levels[sku] = levels.get(sku, 0) + qty
    return sorted(levels.items())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `ON CONFLICT(sku) DO UPDATE SET qty = qty + excluded.qty`.

```python
def upsert(stock, updates):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE stock (sku TEXT PRIMARY KEY, qty INTEGER)")
    con.executemany("INSERT INTO stock VALUES (?, ?)", stock)
    con.executemany("""INSERT INTO stock VALUES (?, ?)
                       ON CONFLICT(sku) DO UPDATE SET qty = qty + excluded.qty""", updates)
    return con.execute("SELECT * FROM stock ORDER BY sku").fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict: `get(sku, 0) + qty`. That costs O(n) time and O(n) space."

"I can do better: `ON CONFLICT(sku) DO UPDATE SET qty = qty + excluded.qty`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/upsert-stock-levels)

<a id="q17"></a>
### Q17. Pivot Months into Columns

🟡 Interview standard · *Conditional aggregation*

Return `(city, January revenue, February revenue)` for every city with orders, sorted by city.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `[('Delhi', 120.0, 60.0), ('Pune', 340.0, 710.0)]`

In [ ]:
def city_by_month(customers, orders):
    """
    Return (city, January revenue, February revenue) for every city with orders,
    sorted by city.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(city_by_month, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), [('Delhi', 120.0, 60.0), ('Pune', 340.0, 710.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dicts city → month → total.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SUM(CASE WHEN month = … THEN amount ELSE 0 END)` — one column per month.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Nested dicts city → month → total.

```python
def city_by_month(customers, orders):
    city_of = {c[0]: c[2] for c in customers}
    table = {}
    for o in orders:
        row = table.setdefault(city_of[o[1]], [0, 0])
        if o[3].startswith("2026-01"):
            row[0] += o[2]
        elif o[3].startswith("2026-02"):
            row[1] += o[2]
    return [(city, jan, feb) for city, (jan, feb) in sorted(table.items())]
```

**Cost:** O(n) time, O(cities) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `SUM(CASE WHEN month = … THEN amount ELSE 0 END)` — one column per month.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def city_by_month(customers, orders):
    con = shop_db(customers, orders)
    sql = """SELECT c.city,
               SUM(CASE WHEN strftime('%m', o.day) = '01' THEN o.amount ELSE 0 END),
               SUM(CASE WHEN strftime('%m', o.day) = '02' THEN o.amount ELSE 0 END)
             FROM orders o JOIN customers c ON c.id = o.customer_id
             GROUP BY c.city ORDER BY c.city"""
    return con.execute(sql).fetchall()
```

**Cost:** O(n) time, O(cities) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dicts city → month → total. That costs O(n) time and O(cities) space."

"I can do better: `SUM(CASE WHEN month = … THEN amount ELSE 0 END)` — one column per month. That's O(n) time and O(cities) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/pivot-months-into-columns)

<a id="q18"></a>
### Q18. Does the Query Use the Index?

🟡 Interview standard · *Indexes · EXPLAIN QUERY PLAN*

Run `SELECT * FROM orders WHERE customer_id = 1`. Return `(plan uses an index before, after creating one)` as two booleans.

**Examples**

- `customers = [(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], orders = [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]` → `(False, True)`

In [ ]:
def uses_index(customers, orders):
    """
    Run SELECT * FROM orders WHERE customer_id = 1. Return (plan uses an index
    before, after creating one) as two booleans.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(uses_index, [
    (([(1, 'Asha', 'Pune'), (2, 'Ravi', 'Delhi'), (3, 'Meera', 'Pune'), (4, 'John', 'Mumbai')], [(101, 1, 250.0, '2026-01-05'), (102, 2, 120.0, '2026-01-05'), (103, 1, 90.0, '2026-01-20'), (104, 3, 400.0, '2026-02-02'), (105, 2, 60.0, '2026-02-14'), (106, 1, 310.0, '2026-02-20')]), (False, True)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Reason about it: without an index the database must scan every row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ask SQLite: `EXPLAIN QUERY PLAN` says SCAN (every row) or SEARCH … USING INDEX.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Reason about it: without an index the database must scan every row.

```python
def uses_index(customers, orders):
    has_index = False
    before = has_index
    has_index = True
    return (before, has_index)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Ask SQLite: `EXPLAIN QUERY PLAN` says SCAN (every row) or SEARCH … USING INDEX.

```python
def shop_db(customers, orders):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT)")
    con.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount REAL, day TEXT)")
    con.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
    con.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
    return con


def uses_index(customers, orders):
    con = shop_db(customers, orders)
    sql = "EXPLAIN QUERY PLAN SELECT * FROM orders WHERE customer_id = 1"
    before = "USING INDEX" in " ".join(str(r) for r in con.execute(sql))
    con.execute("CREATE INDEX idx_orders_customer ON orders(customer_id)")
    after = "USING INDEX" in " ".join(str(r) for r in con.execute(sql))
    return (before, after)
```

**Cost:** O(log n) lookups time, O(n) index space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Reason about it: without an index the database must scan every row. That costs O(1) time and O(1) space."

"I can do better: Ask SQLite: `EXPLAIN QUERY PLAN` says SCAN (every row) or SEARCH … USING INDEX. That's O(log n) lookups time and O(n) index space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/does-the-query-use-the-index)

<a id="q19"></a>
### Q19. All-or-Nothing Transfers

🔴 Stretch · *Transactions · rollback*

Accounts start with `balances`. Apply each transfer (from, to, amount) as one transaction: if it would make a balance negative, roll it back. Return final balances sorted by account.

**Examples**

- `balances = {'A': 100, 'B': 50}, moves = [('A', 'B', 30), ('B', 'A', 500), ('B', 'A', 80)]` → `[('A', 150), ('B', 0)]`

In [ ]:
def transfers(balances, moves):
    """
    Accounts start with balances. Apply each transfer (from, to, amount) as one
    transaction: if it would make a balance negative, roll it back. Return final
    balances sorted by account.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(transfers, [
    (({'A': 100, 'B': 50}, [('A', 'B', 30), ('B', 'A', 500), ('B', 'A', 80)]), [('A', 150), ('B', 0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check the balance first, then move the money in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `CHECK (balance >= 0)` constraint plus `with con:`, which commits or rolls back both updates together.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Check the balance first, then move the money in a dict.

```python
def transfers(balances, moves):
    bal = dict(balances)
    for src, dst, amount in moves:
        if bal[src] - amount >= 0:
            bal[src] -= amount
            bal[dst] += amount
    return sorted(bal.items())
```

**Cost:** O(n) time, O(a) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A `CHECK (balance >= 0)` constraint plus `with con:`, which commits or rolls back both updates together.

```python
def transfers(balances, moves):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE acct (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
    con.executemany("INSERT INTO acct VALUES (?, ?)", balances.items())
    con.commit()
    for src, dst, amount in moves:
        try:
            with con:
                con.execute("UPDATE acct SET balance = balance + ? WHERE name = ?", (amount, dst))
                con.execute("UPDATE acct SET balance = balance - ? WHERE name = ?", (amount, src))
        except sqlite3.IntegrityError:
            pass
    return con.execute("SELECT * FROM acct ORDER BY name").fetchall()
```

**Cost:** O(n) time, O(a) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check the balance first, then move the money in a dict. That costs O(n) time and O(a) space."

"I can do better: A `CHECK (balance >= 0)` constraint plus `with con:`, which commits or rolls back both updates together. That's O(n) time and O(a) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/all-or-nothing-transfers)

<a id="q20"></a>
### Q20. Keep the Latest Version of Each Record

🔴 Stretch · *De-duplication*

`events` are (user\_id, updated\_at, email) rows with repeats. Keep each user's most recent row; return them sorted by user.

**Examples**

- `events = [(1, '2026-01-01', 'a@x.in'), (2, '2026-01-03', 'b@x.in'), (1, '2026-02-10', 'a@new.in'), (2, '2026-01-02', 'old@x.in')]` → `[(1, '2026-02-10', 'a@new.in'), (2, '2026-01-03', 'b@x.in')]`

In [ ]:
def latest(events):
    """
    events are (user_id, updated_at, email) rows with repeats. Keep each user's
    most recent row; return them sorted by user.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(latest, [
    (([(1, '2026-01-01', 'a@x.in'), (2, '2026-01-03', 'b@x.in'), (1, '2026-02-10', 'a@new.in'), (2, '2026-01-02', 'old@x.in')],), [(1, '2026-02-10', 'a@new.in'), (2, '2026-01-03', 'b@x.in')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict that keeps the row with the larger timestamp.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC)` and keep row 1.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** A dict that keeps the row with the larger timestamp.

```python
def latest(events):
    best = {}
    for e in events:
        if e[0] not in best or e[1] > best[e[0]][1]:
            best[e[0]] = e
    return [best[k] for k in sorted(best)]
```

**Cost:** O(n) time, O(users) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC)` and keep row 1.

```python
def latest(events):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE ev (user_id INTEGER, updated_at TEXT, email TEXT)")
    con.executemany("INSERT INTO ev VALUES (?, ?, ?)", events)
    sql = """SELECT user_id, updated_at, email FROM (
               SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC) AS rn FROM ev)
             WHERE rn = 1 ORDER BY user_id"""
    return con.execute(sql).fetchall()
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict that keeps the row with the larger timestamp. That costs O(n) time and O(users) space."

"I can do better: `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC)` and keep row 1. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/sql-python/keep-the-latest-version-of-each-record)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. WHERE vs HAVING?</b></summary>

WHERE filters rows before grouping; HAVING filters groups after aggregation, so it can use SUM, COUNT and so on.

</details>

<details>
<summary><b>2. How do you prevent SQL injection?</b></summary>

Always pass values as parameters (`?` or `%s` placeholders), never with string formatting. Validate any identifiers (table or column names) against an allow-list.

</details>

<details>
<summary><b>3. RANK vs DENSE\_RANK vs ROW\_NUMBER?</b></summary>

For values 10, 10, 8: ROW\_NUMBER gives 1, 2, 3; RANK gives 1, 1, 3; DENSE\_RANK gives 1, 1, 2.

</details>

<details>
<summary><b>4. When should you add an index?</b></summary>

On columns used often in WHERE, JOIN and ORDER BY on large tables, especially when queries return a small share of rows. Not on tiny tables, or on columns that are written far more than read.

</details>

<details>
<summary><b>5. Should you aggregate in SQL or in pandas?</b></summary>

Filter and aggregate in the database, so only the small result travels over the network and into memory. Use pandas for the last mile: reshaping, charts and modelling.

</details>

<details>
<summary><b>6. What is a transaction?</b></summary>

A group of statements that either all take effect (commit) or none do (rollback), like debiting one account and crediting another.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why analysts need SQL as well as pandas
- [ ] I can explain: Parameters, and why f-strings are dangerous
- [ ] I can explain: Joins and aggregation
- [ ] I can explain: Subqueries and window functions
- [ ] I can explain: Dates and pivots in SQL
- [ ] I can explain: SQL ↔ pandas, upserts, transactions and indexes

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Upsert Stock Levels
- [ ] [Q17](#q17) Pivot Months into Columns
- [ ] [Q18](#q18) Does the Query Use the Index?
- [ ] [Q19](#q19) All-or-Nothing Transfers
- [ ] [Q20](#q20) Keep the Latest Version of Each Record

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.